In [130]:
# Import libraries
import pandas as pd
import numpy as np
import sqlalchemy as sql
import json
import pymssql

In [131]:
year = 2050

In [132]:
# Clear this before publishing
engine = sql.create_engine('mssql+pymssql://DDAMWSQL16/demographic_warehouse')
engine2 = sql.create_engine('mssql+pymssql://sql2014b8/GeoDepot')

In [133]:
# settings
pd.set_option('display.max_columns', None)
%matplotlib inline

pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [134]:
xls = pd.ExcelFile(r'T:\socioec\population sim\region_controls\remi_output_for_popsim.xlsx')
df1 = pd.read_excel(xls,  '1b. Summary Forecast_Production' )
df1.head()

,Industry,2022,2026,2029,2032,2035,2040,2050,2060
0,"Forestry, fishing, and hunting",9.119,8.947,9.079,9.368,9.603,9.733,9.615,9.441
1,Mining,1.250,1.290,1.309,1.325,1.325,1.315,1.266,1.223
2,Utilities,5.186,4.980,4.813,4.690,4.556,4.280,3.645,3.050
3,Construction,71.263,70.396,72.894,75.706,76.357,77.051,73.604,69.884
4,Manufacturing,135.163,130.086,131.876,121.088,128.868,141.323,157.522,171.409


In [135]:
# Race for popsim
def get_indxwalk(row):
        if row['Industry'] in ['State and Local Government', 'Federal Civilian']:
            return 'job_01'
        elif row['Industry'] in ['Federal Military']:
            return 'job_02'
        elif row['Industry'] in ['Forestry, fishing, and hunting', 'Farm', 'Mining']:
            return 'job_03'
        elif row['Industry'] in ['Information',
                                'Professional, scientific, and technical services', 
                                 'Administrative, support, waste management, and remediation services']:
            return 'job_04'
        elif row['Industry'] in ['Finance and insurance','Real estate and rental and leasing',
                                 'Management of companies and enterprises']:
            return 'job_05'        
        elif row['Industry'] in ['Educational services; private']:
            return 'job_06'
        elif row['Industry'] in ['Health care and social assistance']:
            return 'job_07'
        elif row['Industry'] in ['Retail trade']:
            return 'job_08'
        elif row['Industry'] in ['Construction','Transportation and warehousing' ]:
            return 'job_09'
        elif row['Industry'] in ['Utilities','Manufacturing', 'Wholesale trade']:
            return 'job_10'
        elif row['Industry'] in ['Arts, entertainment, and recreation']:
            return 'job_11'
        elif row['Industry'] in ['Accommodation']:
            return 'job_12'
        elif row['Industry'] in ['Food Service']:
            return 'job_13'
        elif row['Industry'] in ['Other services (except public administration)']:
            return 'job_14'


df1['job_cat'] = df1.apply(get_indxwalk, axis = 1)
df1


,Industry,2022,2026,2029,2032,2035,2040,2050,2060,job_cat
0,"Forestry, fishing, and hunting",9.119,8.947,9.079,9.368,9.603,9.733,9.615,9.441,job_03
1,Mining,1.250,1.290,1.309,1.325,1.325,1.315,1.266,1.223,job_03
2,Utilities,5.186,4.980,4.813,4.690,4.556,4.280,3.645,3.050,job_10
3,Construction,71.263,70.396,72.894,75.706,76.357,77.051,73.604,69.884,job_09
4,Manufacturing,135.163,130.086,131.876,121.088,128.868,141.323,157.522,171.409,job_10
5,Wholesale trade,36.670,35.463,35.142,35.934,35.673,35.240,33.736,31.374,job_10
6,Retail trade,111.954,102.216,100.137,102.018,101.568,101.074,97.944,92.248,job_08
7,Transportation and warehousing,50.227,54.517,56.039,57.090,57.693,58.549,58.876,58.386,job_09
8,Information,21.267,20.757,20.332,20.278,19.908,19.416,18.430,17.293,job_04
9,Finance and insurance,66.658,66.801,66.226,66.540,66.030,64.527,59.964,54.903,job_05


In [136]:
# TODO: Do this for every increment year (after 2022) 
temp = df1.groupby(['job_cat']).agg({year: 'sum'}).reset_index()
temp['jobs'] = temp[year]*1000

In [137]:
# reading the gQ pop from concep
# We are doing this to remove GQ from the total number of workers 
# TODO: need to change the year 
query_h = f'''
  SELECT sum(gq_mil) FROM [sr15_dev].[capacity_outputs].[mgrabase]
WHERE increment = {year}
'''
gqmil_df = pd.read_sql(query_h, con=engine.connect())
print("gqmil_df shape: ", gqmil_df.shape)
gqmil_jobs = gqmil_df.iloc[0, 0]
print("gqmil_jobs: ", gqmil_jobs)

gqmil_df shape:  (1, 1)
gqmil_jobs:  41181


In [138]:
temp = temp.sort_values('job_cat', ascending = True)
temp_t = temp.set_index('job_cat').transpose()
temp_t.loc['jobs', 'job_02'] = temp_t.loc['jobs', 'job_02'] - gqmil_jobs
temp_t['region'] = 1
temp_t.set_index('region', inplace = True)
temp_t.columns.name = None
temp_t

,job_01,job_02,job_03,job_04,job_05,job_06,job_07,job_08,job_09,job_10,job_11,job_12,job_13,job_14
region,,,,,,,,,,,,,,
1,105.708,104.000,10.882,315.494,147.589,138.929,192.759,97.944,132.480,194.903,48.810,15.235,132.477,73.725
1,105708.420,62819.000,10881.672,315493.877,147588.561,138929.177,192759.014,97944.394,132480.052,194903.377,48810.367,15234.934,132477.198,73724.925


In [139]:
output = pd.DataFrame(temp_t.iloc[1,:]).T
output= output.round()
output = output.astype(int)
output['region'] = 1
output = output[['region','job_01', 'job_02', 'job_03', 'job_04', 'job_05', 'job_06', 'job_07',
       'job_08', 'job_09', 'job_10', 'job_11', 'job_12', 'job_13', 'job_14']]
output


,region,job_01,job_02,job_03,job_04,job_05,job_06,job_07,job_08,job_09,job_10,job_11,job_12,job_13,job_14
1,1,105708,62819,10882,315494,147589,138929,192759,97944,132480,194903,48810,15235,132477,73725


# Appending Labor Force Components

In [140]:
lf_comp = pd.read_excel(xls,  '2. Labor Force Components' )
lf_comp = lf_comp[lf_comp['Race'] != 'All Races']
lf_comp.head()

,Category,Race,Units,2022,2026,2029,2032,2035,2040,2050,2060
1,Labor Force,White-NonHispanic,Thousands,652.608,643.159,638.758,637.175,637.193,637.604,627.850,619.389
2,Labor Force,Black-NonHispanic,Thousands,57.959,56.416,55.923,55.801,55.926,56.340,55.685,54.768
3,Labor Force,Other-NonHispanic,Thousands,277.929,299.097,313.579,327.614,340.734,359.425,382.280,391.228
4,Labor Force,Hispanic,Thousands,506.340,521.019,530.458,538.257,544.572,547.959,541.121,528.512


In [141]:
# Change nameing convention 
# Let's say you want to rename 'White-NonHispanic' to 'White', 'Black-NonHispanic' to 'Black', etc.
new_race_names = {
    'White-NonHispanic': 'lfp_white',
    'Black-NonHispanic': 'lfp_black',
    'Other-NonHispanic': 'lfp_other',
    'Hispanic': 'lfp_hispanic'
}

# Apply the renaming
lf_comp['Race'] = lf_comp['Race'].map(new_race_names)


In [142]:
lf_comp = lf_comp.drop(['Category', 'Units'], axis=1)
lf_comp

,Race,2022,2026,2029,2032,2035,2040,2050,2060
1,lfp_white,652.608,643.159,638.758,637.175,637.193,637.604,627.850,619.389
2,lfp_black,57.959,56.416,55.923,55.801,55.926,56.340,55.685,54.768
3,lfp_other,277.929,299.097,313.579,327.614,340.734,359.425,382.280,391.228
4,lfp_hispanic,506.340,521.019,530.458,538.257,544.572,547.959,541.121,528.512


In [143]:
# Resetting the index to turn the 'Race' into a normal column
lf_comp_reset = lf_comp.reset_index().rename(columns={'index': 'Year'})

# Pivoting the DataFrame
lf_comp_pivoted = lf_comp_reset.melt(id_vars=['Race'], var_name='Year', value_name='Value')
lf_comp_pivoted = lf_comp_pivoted.pivot(index='Year', columns='Race', values='Value')

lf_comp_pivoted = lf_comp_pivoted*1000
lf_comp_pivoted

Race,lfp_black,lfp_hispanic,lfp_other,lfp_white
Year,,,,
2022,57959.133,506340.092,277929.102,652607.798
2026,56416.173,521018.895,299097.020,643158.628
2029,55922.872,530458.047,313578.521,638757.513
2032,55801.384,538256.840,327614.484,637174.891
2035,55925.522,544571.707,340733.983,637192.761
2040,56339.999,547958.672,359424.573,637604.246
2050,55684.842,541120.556,382280.071,627850.499
2060,54768.083,528512.486,391228.198,619389.150
Year,2000.000,4000.000,3000.000,1000.000


In [144]:
combined_output = pd.concat([output.reset_index(drop=True), lf_comp_pivoted.loc[[year]].reset_index(drop=True)], axis=1).astype(int)

In [145]:
combined_output.to_csv(fr'outputs/regional_control_ind_{year}.csv', index=False)